# U-MIMIC: Unified Mechanistic Inference from Multimodal Imaging and Counts

**Quickstart Notebook** — A guided tour of U-MIMIC's core capabilities.

This notebook covers:
1. **Setup** — Install and configure
2. **Cell Dynamics** — Define cell states, rates, and dose-response functions
3. **Deterministic Simulation** — ODE-based population trajectories
4. **Stochastic Simulation** — Gillespie SSA with ensemble analysis
5. **Dose-Response Analysis** — Cytostatic vs. cytotoxic mechanism decomposition
6. **Synthetic Data Generation** — Simulate realistic experimental datasets
7. **Parameter Inference (MLE)** — Recover parameters from noisy data
8. **Public Datasets** — Load and analyze real experimental data
9. **Pharmacokinetics** — In vivo drug exposure modeling
10. **Pipeline** — YAML-driven experiment configuration

---
## 1. Setup

In [ ]:
# Install U-MIMIC (run once)
# !pip install -e /path/to/U-MIMIC

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print("Setup complete!")

---
## 2. Cell Dynamics: States, Rates, and Dose-Response

U-MIMIC models cancer cell populations as a system of interacting states:
- **P** (Proliferating) — Dividing cells
- **Q** (Quiescent) — Resting cells
- **A** (Apoptotic) — Dying cells
- **R** (Resistant) — Drug-resistant cells

Drugs affect the *rates* (birth, death, transitions) as functions of concentration.

In [ ]:
from umimic.dynamics.states import CellType, ModelTopology

# Standard 2-state model: Proliferating <-> Quiescent
topology = ModelTopology.two_state()
print(f"States: {[ct.name for ct in topology.active_states]}")
print(f"Transitions: {[(s.name, t.name) for s, t in topology.transitions]}")
print(f"Division states: {[ct.name for ct in topology.division_states]}")
print(f"Death states: {[ct.name for ct in topology.death_states]}")
print(f"Number of states: {topology.n_states}")

In [ ]:
from umimic.dynamics.rates import RateSet, EmaxHill

# Define a cytotoxic drug: increases death rate with concentration
cytotoxic_rates = RateSet.cytotoxic_drug(
    b0=0.04,          # birth rate: 0.04/h (doubling time ~17h)
    d0=0.01,          # baseline death rate
    emax_death=0.06,  # max drug-induced death rate increase
    ec50_death=2.0,   # half-max concentration
    hill_death=1.5,   # Hill coefficient (steepness)
)

# Inspect rates at different concentrations
print("Concentration | Birth | Death(P) | Net Growth")
print("-" * 50)
for c in [0, 0.5, 1, 2, 5, 10]:
    b = cytotoxic_rates.birth_rate(c)
    d = cytotoxic_rates.death_rate(CellType.P, c)
    ng = cytotoxic_rates.net_growth_rate(c)
    print(f"  {c:>10.1f}   | {b:.4f} | {d:.4f}   | {ng:+.4f}")

In [ ]:
# Custom rate set with explicit drug effects on both birth and death
custom_rates = RateSet(
    birth_base=0.04,
    birth_modulation=EmaxHill(emax=0.8, ec50=3.0, hill=1.2),  # cytostatic component
    death_base={CellType.P: 0.01, CellType.Q: 0.005},
    death_modulation={
        CellType.P: EmaxHill(emax=0.05, ec50=2.0, hill=1.5),  # cytotoxic component
    },
    transition_base={
        (CellType.P, CellType.Q): 0.005,  # arrest rate
        (CellType.Q, CellType.P): 0.003,  # reawakening rate
    },
)
print("Custom mixed-mechanism rate set created!")
print(f"Birth at C=0: {custom_rates.birth_rate(0):.4f}")
print(f"Birth at C=10: {custom_rates.birth_rate(10):.4f}  (reduced = cytostatic)")
print(f"Death(P) at C=0: {custom_rates.death_rate(CellType.P, 0):.4f}")
print(f"Death(P) at C=10: {custom_rates.death_rate(CellType.P, 10):.4f}  (increased = cytotoxic)")

---
## 3. Deterministic Simulation (ODE)

The mean-field ODE system describes how cell populations evolve over time:

$$\frac{dP}{dt} = b(C) \cdot P - d_P(C) \cdot P - u_{PQ}(C) \cdot P + u_{QP}(C) \cdot Q$$
$$\frac{dQ}{dt} = u_{PQ}(C) \cdot P - u_{QP}(C) \cdot Q - d_Q(C) \cdot Q$$

In [ ]:
from umimic.dynamics.ode_system import CellDynamicsODE
from umimic.visualization.trajectories import plot_population_trajectories

# Simulate at zero drug (control)
exposure_fn = lambda t: 0.0  # constant zero concentration
ode = CellDynamicsODE(cytotoxic_rates, topology, exposure_fn)

y0 = np.array([200.0, 0.0])  # 200 proliferating cells, 0 quiescent
t_eval = np.linspace(0, 96, 200)  # 96 hours

result_ctrl = ode.solve(y0, (0, 96), t_eval)

fig = plot_population_trajectories(
    result_ctrl,
    title="Control: Untreated Cell Growth"
)
plt.show()

In [ ]:
# Dose-response: simulate across multiple concentrations
from umimic.visualization.trajectories import plot_dose_response_trajectories

concentrations = [0, 0.5, 1.0, 2.0, 5.0, 10.0]
dose_results = ode.solve_dose_response(y0, (0, 96), concentrations, t_eval)

fig = plot_dose_response_trajectories(
    dose_results,
    state="P",
    title="Proliferating Cells Under Cytotoxic Drug"
)
plt.show()

# Show total viable cells
fig, ax = plt.subplots()
for conc in concentrations:
    r = dose_results[conc]
    label = f"C={conc}" if conc > 0 else "Control"
    ax.plot(r.times, r.viable, label=label)
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Total viable cells")
ax.set_title("Viable Cell Count vs. Drug Concentration")
ax.legend(title="Concentration")
plt.tight_layout()
plt.show()

---
## 4. Stochastic Simulation (Gillespie SSA)

For small populations or when variability matters, U-MIMIC provides exact stochastic simulation via the Gillespie algorithm. The ensemble captures intrinsic noise in cell birth/death/transition events.

In [ ]:
from umimic.dynamics.gillespie import GillespieSimulator
from umimic.visualization.trajectories import plot_ensemble

rng = np.random.default_rng(42)
exposure_fn = lambda t: 0.0

sim = GillespieSimulator(cytotoxic_rates, topology, exposure_fn, rng=rng)

# Single trajectory
y0_int = np.array([100.0, 0.0])  # start with 100 cells
t_record = np.linspace(0, 72, 50)

single_result = sim.simulate(y0_int, t_max=72, t_record=t_record)
fig = plot_population_trajectories(
    single_result,
    title="Single Gillespie Trajectory (Control)"
)
plt.show()

In [ ]:
# Ensemble of 50 stochastic trajectories
ensemble = sim.simulate_ensemble(
    y0_int, t_max=72, t_record=t_record, n_trajectories=50
)

fig = plot_ensemble(
    ensemble,
    states=["P"],
    show_mean=True,
    show_ci=True,
    alpha_traj=0.08,
    title="Gillespie Ensemble: Proliferating Cells (n=50)"
)
plt.show()

# Compare mean and variance
means = ensemble.mean()
stds = ensemble.std()
print(f"At t=72h: P = {means['P'][-1]:.0f} +/- {stds['P'][-1]:.0f}")

In [ ]:
# Stochastic simulation under drug treatment
exposure_drug = lambda t: 5.0  # constant 5 uM
sim_drug = GillespieSimulator(cytotoxic_rates, topology, exposure_drug, rng=rng)

ensemble_drug = sim_drug.simulate_ensemble(
    y0_int, t_max=72, t_record=t_record, n_trajectories=50
)

# Compare control vs. drug
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

plot_ensemble(ensemble, states=["P", "Q"], ax=axes[0],
              title="Control (C=0)")
plot_ensemble(ensemble_drug, states=["P", "Q"], ax=axes[1],
              title="Drug Treatment (C=5 uM)")

plt.tight_layout()
plt.show()

---
## 5. Dose-Response Analysis

U-MIMIC decomposes drug effects into mechanistic components:
- **Cytostatic**: Drug reduces birth rate (proliferation arrest)
- **Cytotoxic**: Drug increases death rate (cell killing)

This provides richer insight than traditional IC50 curves.

In [ ]:
from umimic.visualization.dose_response import (
    plot_rate_dose_response,
    plot_net_growth_curve,
    plot_mechanism_comparison,
)

# Rate dose-response: how individual rates change with concentration
fig = plot_rate_dose_response(
    cytotoxic_rates,
    rates_to_plot=["birth", "death_P", "net_growth"],
    title="Cytotoxic Drug: Rate Dose-Response"
)
plt.show()

In [ ]:
# Net growth curve with NG0 and NG50 annotations
# NG0 = concentration where growth stops (mechanistic alternative to IC50)
# NG50 = concentration where growth is halved
fig = plot_net_growth_curve(
    cytotoxic_rates,
    title="Net Growth Rate with NG0 and NG50"
)
plt.show()

In [ ]:
# Compare cytostatic vs. cytotoxic mechanisms
from umimic.dynamics.rates import RateSet

cytostatic_rates = RateSet.cytostatic_drug(
    b0=0.04, d0=0.01,
    emax_birth=0.8,    # 80% max reduction in birth rate
    ec50_birth=1.0,
    hill_birth=1.5,
)

# Each call creates a 2-panel figure (absolute rates + fold-change)
print("--- Cytotoxic Drug ---")
fig = plot_mechanism_comparison(cytotoxic_rates, title="Cytotoxic Drug")
plt.show()

print("\n--- Cytostatic Drug ---")
fig = plot_mechanism_comparison(cytostatic_rates, title="Cytostatic Drug")
plt.show()

---
## 6. Synthetic Data Generation

Generate realistic in vitro dose-response datasets with observation noise (negative binomial cell counts), suitable for testing inference pipelines.

In [ ]:
from umimic.data.synthetic import SyntheticDataGenerator
from umimic.observations.cell_counts import CellCountObservation

# Set up generator with known "true" parameters
obs_model = CellCountObservation(overdispersion=15.0)
gen = SyntheticDataGenerator(
    cytotoxic_rates, topology, obs_model,
    rng=np.random.default_rng(42),
)

# Generate a synthetic dose-response plate
dataset = gen.generate_invitro_plate(
    initial_cells=np.array([200.0, 0.0]),
    concentrations=[0, 0.3, 1.0, 3.0, 10.0],
    n_wells_per_dose=3,
    t_max=72.0,
    dt_obs=4.0,
    method="ode",
)

print(f"Generated dataset: {dataset.n_series} series")
print(f"Concentrations: {dataset.concentrations}")
print(f"Time points per series: {dataset.series[0].n_timepoints}")

In [ ]:
# Visualize the synthetic data
fig, axes = plt.subplots(1, len(dataset.concentrations), figsize=(18, 4), sharey=True)

for i, conc in enumerate(dataset.concentrations):
    series_at_conc = dataset.by_concentration(conc)
    for s in series_at_conc:
        axes[i].plot(s.times, s.observations['cell_counts'], 'o-',
                    alpha=0.6, markersize=2)
    axes[i].set_title(f"C = {conc}")
    axes[i].set_xlabel("Time (h)")
    if i == 0:
        axes[i].set_ylabel("Cell count")

plt.suptitle("Synthetic In Vitro Dose-Response Data", fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Parameter Inference (MLE)

Estimate the mechanistic parameters from noisy data using Maximum Likelihood Estimation. U-MIMIC computes the log-likelihood by solving the ODE forward and evaluating observation probabilities.

In [ ]:
from umimic.inference.likelihood import ModelLikelihood
from umimic.inference.mle import MLEstimator
from umimic.inference.priors import PriorSpec

# Set up the likelihood function
param_names = [
    "b0", "d0_P", "emax_death", "ec50_death",
    "hill_death", "u_PQ", "u_QP", "overdispersion",
]

likelihood = ModelLikelihood(
    topology=topology,
    data=dataset.series,
    param_names=param_names,
    mode="ode",
)

print(f"Number of parameters: {likelihood.n_params}")
print(f"Parameter names: {param_names}")
print(f"Data series: {len(dataset.series)}")

In [ ]:
# Fit the model
priors = PriorSpec.default_invitro()
estimator = MLEstimator(likelihood, priors=priors, method="L-BFGS-B")

print("Fitting model (multi-start optimization)...")
result = estimator.fit(n_restarts=3)

print(f"\nConverged: {result.converged}")
print(f"Log-likelihood: {result.log_likelihood:.2f}")
print(f"AIC: {result.aic:.2f}")
print(f"Evaluations: {result.n_evaluations}")
print(f"\nEstimated parameters:")
print(f"{'Parameter':<18} {'Estimated':>10} {'True':>10}")
print("-" * 40)

true_values = {
    'b0': 0.04, 'd0_P': 0.01, 'emax_death': 0.06,
    'ec50_death': 2.0, 'hill_death': 1.5,
    'u_PQ': 0.005, 'u_QP': 0.003, 'overdispersion': 15.0
}
for name, est in result.parameters.items():
    true_val = true_values.get(name, '?')
    print(f"{name:<18} {est:>10.4f} {true_val:>10}")

In [ ]:
# Visualize: compare model prediction at estimated params vs. data
from umimic.dynamics.ode_system import CellDynamicsODE

est_rates = RateSet.cytotoxic_drug(
    b0=result.parameters['b0'],
    d0=result.parameters['d0_P'],
    emax_death=result.parameters['emax_death'],
    ec50_death=result.parameters['ec50_death'],
    hill_death=result.parameters['hill_death'],
)

fig, axes = plt.subplots(1, len(dataset.concentrations), figsize=(18, 4), sharey=True)

for i, conc in enumerate(dataset.concentrations):
    # Plot data
    for s in dataset.by_concentration(conc):
        axes[i].plot(s.times, s.observations['cell_counts'], 'o',
                    alpha=0.4, color='gray', markersize=3)
    
    # Plot model prediction
    ode_fit = CellDynamicsODE(est_rates, topology, lambda t, c=conc: c)
    pred = ode_fit.solve(y0, (0, 72), t_eval=np.linspace(0, 72, 100))
    axes[i].plot(pred.times, pred.viable, 'r-', linewidth=2, label='Model fit')
    
    axes[i].set_title(f"C = {conc}")
    axes[i].set_xlabel("Time (h)")
    if i == 0:
        axes[i].set_ylabel("Cell count")

axes[-1].legend()
plt.suptitle("MLE Model Fit vs. Synthetic Data", fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Public Datasets: Real Experimental Data

U-MIMIC includes loaders for publicly available cancer drug response datasets.

**PhenoPop dataset**: Ba/F3 cells (sensitive + resistant) under imatinib.
- 11 concentrations (0–5 uM)
- 14 time points (every 3 hours)
- Mixture and monoclonal populations

First, download the data:
```bash
python data/public/download_datasets.py --dataset phenopop
python data/public/convert_phenopop.py
```

In [ ]:
from umimic.data.public_datasets import load_phenopop, list_available_datasets

# Check what's available
datasets_info = list_available_datasets()
for name, info in datasets_info.items():
    status = "READY" if info['downloaded'] == 'yes' else 'not downloaded'
    print(f"  [{status:>14s}] {name}: {info['description']}")

In [ ]:
# Load PhenoPop mixture data
try:
    phenopop = load_phenopop(population='mix')
    print(f"Loaded: {phenopop.n_series} series")
    print(f"Concentrations: {phenopop.concentrations}")
    print(f"Cell line: {phenopop.series[0].metadata.get('cell_line', 'N/A')}")
    print(f"Drug: {phenopop.series[0].metadata.get('drug', 'N/A')}")
except FileNotFoundError:
    print("PhenoPop data not downloaded yet.")
    print("Run: python data/public/download_datasets.py --dataset phenopop")
    print("Then: python data/public/convert_phenopop.py")
    phenopop = None

In [ ]:
# Visualize PhenoPop dose-response
if phenopop is not None:
    # Pick one mixture ratio to visualize
    concs_to_show = [0.0, 0.125, 0.5, 2.5, 5.0]
    fig, axes = plt.subplots(1, len(concs_to_show), figsize=(18, 4), sharey=True)
    
    for i, conc in enumerate(concs_to_show):
        series = phenopop.by_concentration(conc)
        for s in series[:6]:  # show up to 6 replicates
            if 'mix_1to1' in (s.metadata.get('population', '') or s.group_id):
                axes[i].plot(s.times, s.observations['cell_counts'],
                            'o-', alpha=0.5, markersize=3)
        axes[i].set_title(f"Imatinib {conc} uM")
        axes[i].set_xlabel("Time (h)")
        if i == 0:
            axes[i].set_ylabel("Cell count")
    
    plt.suptitle("PhenoPop: Ba/F3 1:1 Mixture Under Imatinib", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Dose-response summary: final cell count vs. concentration
if phenopop is not None:
    concs = []
    means = []
    sds = []

    for conc in phenopop.concentrations:
        series = phenopop.by_concentration(conc)
        # Filter to BF_11 (1:1 mix) only
        finals = [s.observations['cell_counts'][-1] for s in series
                  if 'mix_1to1' in (s.metadata.get('population', '') or s.group_id)]
        if finals:
            concs.append(conc)
            means.append(np.mean(finals))
            sds.append(np.std(finals))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(concs, means, yerr=sds, fmt='o-', capsize=4, color='#2196F3')
    ax.set_xlabel('Imatinib Concentration (uM)')
    ax.set_ylabel('Final Cell Count (t=39h)')
    ax.set_title('PhenoPop: Dose-Response (Ba/F3 1:1 Mix)')
    ax.set_xscale('symlog', linthresh=0.01)
    plt.tight_layout()
    plt.show()

---
## 9. Pharmacokinetics: In Vivo Drug Exposure

For in vivo modeling, drug concentration varies over time according to pharmacokinetic (PK) models. U-MIMIC provides 1- and 2-compartment PK models that drive the exposure function.

In [ ]:
from umimic.pk.compartment import OneCompartmentPK, TwoCompartmentPK
from umimic.pk.dosing import DosingSchedule, Dose
from umimic.pk.exposure import ExposureProfile

# 1-compartment PK model with oral dosing
pk_model = OneCompartmentPK(vd=10.0, ke=0.1, ka=0.5)
print(f"PK half-life: {pk_model.half_life:.1f} hours")

# Dosing schedule: 100 mg every 24h for 5 days
schedule = DosingSchedule([
    Dose(time=i*24, amount=100.0, route="oral")
    for i in range(5)
])

# Create exposure profile
exposure = ExposureProfile.from_pk(pk_model, schedule)

# Plot PK profile
t = np.linspace(0, 168, 500)  # 7 days
c = np.array([exposure.concentration(ti) for ti in t])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t / 24, c, color='#2196F3', linewidth=2)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Plasma Concentration')
ax.set_title('1-Compartment PK: Repeated Oral Dosing (100 mg q24h x 5)')
for dose in schedule.doses:
    ax.axvline(x=dose.time/24, color='gray', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Simulate tumor growth under PK-driven drug exposure
ode_invivo = CellDynamicsODE(cytotoxic_rates, topology, exposure)

y0_tumor = np.array([1000.0, 0.0])  # larger initial tumor
t_eval_iv = np.linspace(0, 168, 300)  # 7 days

result_treated = ode_invivo.solve(y0_tumor, (0, 168), t_eval_iv)

# Compare with untreated control
ode_ctrl = CellDynamicsODE(cytotoxic_rates, topology, lambda t: 0.0)
result_untreated = ode_ctrl.solve(y0_tumor, (0, 168), t_eval_iv)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                          gridspec_kw={'height_ratios': [1, 2]})

# Top: PK profile
axes[0].plot(t_eval_iv / 24, [exposure.concentration(ti) for ti in t_eval_iv],
            color='#FF9800', linewidth=1.5)
axes[0].set_ylabel('Drug Conc.')
axes[0].set_title('In Vivo Simulation: PK-Driven Drug Exposure')

# Bottom: Cell populations
axes[1].plot(result_untreated.times / 24, result_untreated.viable,
            '--', color='gray', label='Untreated')
axes[1].plot(result_treated.times / 24, result_treated.viable,
            '-', color='#F44336', linewidth=2, label='Treated')
axes[1].set_xlabel('Time (days)')
axes[1].set_ylabel('Viable cells')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 10. Pipeline: YAML-Driven Experiments

U-MIMIC supports running complete experiments from YAML configuration files, making analyses reproducible and shareable.

In [ ]:
from umimic.pipeline.config import ExperimentConfig, save_config, load_config
from umimic.pipeline.experiment import Experiment

# Create an experiment configuration
config = ExperimentConfig(
    name="notebook_demo",
    context="in_vitro",
    dynamics={"states": ["P", "Q"]},
    dosing={"concentrations": [0, 0.3, 1.0, 3.0, 10.0]},
    observations={"cell_count_overdispersion": 15.0},
    simulation={"method": "ode", "t_max": 72.0, "dt_obs": 4.0, "n_replicates": 3},
    inference={"mode": "mle", "n_restarts": 3},
)

print("Experiment Configuration:")
print(f"  Name: {config.name}")
print(f"  Context: {config.context}")
print(f"  States: {config.dynamics.states}")
print(f"  Method: {config.simulation.method}")
print(f"  Concentrations: {config.dosing.concentrations}")
print(f"  Inference: {config.inference.mode}")

In [ ]:
# Create and run experiment
exp = Experiment(config)

# Simulate
sim_results = exp.simulate(method="ode", concentrations=[0, 1.0, 10.0])

print(f"Simulation complete! Results for {len(sim_results)} concentrations")
for conc, result in sim_results.items():
    print(f"  C={conc}: final viable = {result.viable[-1]:.0f}")

In [ ]:
# Generate synthetic dataset from the experiment
synth_dataset = exp.generate_synthetic()
print(f"Synthetic dataset: {synth_dataset.n_series} series")
print(f"Concentrations: {synth_dataset.concentrations}")

In [ ]:
# Save config for reproducibility
import tempfile
from pathlib import Path

config_path = Path(tempfile.gettempdir()) / "umimic_demo_config.yaml"
save_config(config, config_path)

# Show the YAML
print(config_path.read_text())

---
## Summary

This notebook covered the core U-MIMIC workflow:

| Step | Module | Description |
|------|--------|-------------|
| **Define topology** | `dynamics.states` | Choose cell states (P, Q, A, R) and transitions |
| **Set rates** | `dynamics.rates` | Birth/death/transition rates with dose-response |
| **Simulate (ODE)** | `dynamics.ode_system` | Deterministic mean-field trajectories |
| **Simulate (SSA)** | `dynamics.gillespie` | Stochastic individual-cell simulation |
| **Observe** | `observations.cell_counts` | Noisy observations (negative binomial) |
| **Generate data** | `data.synthetic` | Create realistic synthetic datasets |
| **Infer** | `inference.mle` | Recover parameters from data |
| **Real data** | `data.public_datasets` | Load PhenoPop, TSHS, BESTDR, etc. |
| **PK/PD** | `pk.compartment` | In vivo drug exposure modeling |
| **Pipeline** | `pipeline.experiment` | YAML-driven reproducible experiments |

### Next Steps

- **MCMC inference**: `umimic.inference.mcmc` for full posterior distributions
- **Hierarchical models**: `umimic.inference.hierarchical` for population-level inference
- **BLI / tumor volume**: `umimic.observations.bli`, `umimic.observations.tumor_volume`
- **Interactive dashboard**: `streamlit run dashboard/app.py`
- **Transfer learning**: `umimic.pipeline.transfer` for in vitro -> in vivo parameter transfer